## Desenvolvimento de dimensoes e fatos na modelagem estrela
#### Dimensoes: elas guardam os atributos descritivos que dão contexto aos fatos (quem, o quê, onde, quando), e cada uma tem uma chave primária (normalmente uma surrogate key)

- dim_date
- dim_customer
- dim_product
- dim_store
- dim_payment

In [0]:
%sql
CREATE OR REPLACE TABLE proj_dw.dw_star.dim_date (
    date_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    date_value DATE,
    year INT,
    month INT,
    day INT,
    month_name STRING
);

INSERT INTO proj_dw.dw_star.dim_date (date_value, year, month, day, month_name)

SELECT DISTINCT
    order_date AS date_value,
    YEAR(order_date),
    MONTH(order_date),
    DAY(order_date),
    DATE_FORMAT(order_date, 'MMMM')

FROM bronze.sales_orders

WHERE order_date IS NOT NULL;


In [0]:
%sql
SELECT *
FROM proj_dw.dw_star.dim_date
LIMIT 10

In [0]:
%sql
CREATE OR REPLACE TABLE proj_dw.dw_star.dim_customer (
  customer_sk BIGINT GENERATED ALWAYS AS IDENTITY, 
  customer_id STRING,
  customer_name STRING,
  customer_email STRING,
  customer_birthdate DATE,
  customer_gender STRING,
  customer_city STRING,
  customer_state STRING
);

INSERT INTO proj_dw.dw_star.dim_customer (
  customer_id,
  customer_name, 
  customer_email,
  customer_birthdate, 
  customer_gender,
  customer_city, 
  customer_state
)

SELECT DISTINCT
  customer_id, 
  customer_name, 
  customer_email,
  customer_birthdate, 
  customer_gender,
  customer_city, 
  customer_state

FROM bronze.sales_orders;

In [0]:
%sql
SELECT *
FROM proj_dw.dw_star.dim_customer
LIMIT 10

In [0]:
%sql
CREATE OR REPLACE TABLE proj_dw.dw_star.dim_product (
  product_sk BIGINT GENERATED ALWAYS AS IDENTITY,
  product_id STRING,
  product_name STRING,
  product_category STRING,
  product_subcategory STRING,
  brand STRING
);

INSERT INTO proj_dw.dw_star.dim_product (
  product_id, 
  product_name, 
  product_category,
  product_subcategory, 
  brand
)

SELECT DISTINCT
  product_id, 
  product_name, 
  product_category,
  product_subcategory, 
  brand

FROM bronze.sales_orders;

In [0]:
%sql
SELECT *
FROM proj_dw.dw_star.dim_product
LIMIT 5

-- Perceba que no modelo estrela, categoria, subcategoria e marca estao vindo tudo na tabela de produtos

In [0]:
%sql
CREATE OR REPLACE TABLE proj_dw.dw_star.dim_store (
  store_sk   BIGINT GENERATED ALWAYS AS IDENTITY,
  store_id   STRING,
  store_name STRING,
  store_city STRING,
  store_state STRING
);

INSERT INTO proj_dw.dw_star.dim_store (
  store_id,
  store_name,
  store_city,
  store_state
)

SELECT DISTINCT
  store_id,
  store_name,
  store_city,
  store_state

FROM bronze.sales_orders;


In [0]:
%sql
SELECT *
FROM proj_dw.dw_star.dim_store
LIMIT 5

In [0]:
%sql
CREATE OR REPLACE TABLE proj_dw.dw_star.dim_payment (
  payment_sk          BIGINT GENERATED ALWAYS AS IDENTITY,
  payment_method      STRING,
  payment_installments INT,
  payment_status      STRING
);

INSERT INTO proj_dw.dw_star.dim_payment (
  payment_method,
  payment_installments,
  payment_status
)

SELECT DISTINCT
  payment_method,
  payment_installments,
  payment_status

FROM proj_dw.bronze.sales_orders;

In [0]:
%sql
SELECT *
FROM proj_dw.dw_star.dim_payment
LIMIT 5

In [0]:
%sql
SELECT *
FROM proj_dw.dw_star.dim_payment
WHERE payment_method = 'Cartão de Crédito'
LIMIT 5

## Fatos: guarda os eventos de negócio e suas métricas numéricas.

- fact_order_items

In [0]:
%sql
USE CATALOG proj_dw;

CREATE OR REPLACE TABLE dw_star.fact_order_items AS
SELECT
  b.order_id,
  b.order_item_id,

  -- chaves substitutas que estao vindo das dimensoes criadas
  d_date.date_sk         AS order_date_sk,
  d_cust.customer_sk     AS customer_sk,
  d_prod.product_sk      AS product_sk,
  d_store.store_sk       AS store_sk,
  d_pay.payment_sk       AS payment_sk,

  -- métricas numéricas
  b.quantity,
  b.unit_price,
  b.discount_value,
  b.total_gross,
  b.total_net

FROM proj_dw.bronze.sales_orders b

JOIN dw_star.dim_date      d_date
ON b.order_date = d_date.date_value

JOIN dw_star.dim_customer  d_cust
ON b.customer_id = d_cust.customer_id

JOIN dw_star.dim_product   d_prod
ON b.product_id = d_prod.product_id

JOIN dw_star.dim_store     d_store
ON b.store_id = d_store.store_id

JOIN dw_star.dim_payment   d_pay
ON  b.payment_method       = d_pay.payment_method
AND b.payment_installments = d_pay.payment_installments
AND b.payment_status       = d_pay.payment_status;

In [0]:
%sql
SELECT *
FROM proj_dw.dw_star.fact_order_items
LIMIT 5

-- Cada linha é um item do pedido, com as chaves das dimensões (data, cliente, produto, loja, pagamento) e as métricas da venda (quantidade, preço, desconto, total bruto e líquido)

In [0]:
%sql
-- Criando uma tabela agregada analisando o desempenho de categorias de produto, por Estado e ano
USE CATALOG proj_dw;

SELECT
    d_date.year,
    d_prod.product_category,
    d_store.store_state,
    ROUND(SUM(f.total_net), 2) AS revenue

FROM dw_star.fact_order_items f

JOIN dw_star.dim_date d_date ON f.order_date_sk = d_date.date_sk
JOIN dw_star.dim_product d_prod ON f.product_sk = d_prod.product_sk
JOIN dw_star.dim_store d_store ON f.store_sk = d_store.store_sk

GROUP BY
    d_date.year,
    d_prod.product_category,
    d_store.store_state

ORDER BY 
    d_date.year,
    d_prod.product_category,
    d_store.store_state